# Startup Blueprint Generator Agent — Working Code (RAG + IBM Granite)

**Problem Statement No. 20 — Startup Blueprint Generator Agent**
AICTE IBM SkillsBuild Summer Internship Program (SIP) 2026
Student: Anshul Pravin Bhusal | 202501040341@mitaoe.ac.in | MIT Academy of Engineering

This notebook is a standalone, runnable implementation of the same RAG + IBM Granite
architecture configured inside the deployed **watsonx Orchestrate** agent. It demonstrates the
core mechanism end-to-end: retrieving relevant knowledge from the curated startup knowledge
base, then generating a structured 9-section business blueprint using an **IBM Granite** model
via the **watsonx.ai** API — satisfying the mandatory technology requirement (IBM Cloud Lite /
IBM Granite) with fully inspectable, executable code.

**Architecture**
1. Load the 4 curated knowledge base documents (schemes, funding, legal, BMC/GTM templates)
2. Retrieve the most relevant chunks for a user's startup idea (simple keyword-overlap retrieval)
3. Construct a grounded prompt (retrieved context + behavior instructions)
4. Call an IBM Granite model on watsonx.ai to generate the structured blueprint
5. Print the final 9-section business blueprint


## 1. Setup & Credentials

Fill in your IBM Cloud / watsonx.ai credentials below. Get these from your [IBM Cloud Lite account](https://cloud.ibm.com) → watsonx.ai project → Manage → Access Control / Service Credentials.

In [ ]:
# Install required packages (uncomment if running for the first time)
# !pip install ibm-watsonx-ai --quiet

import os
import glob
import re

# ---- Fill in your credentials here ----
WATSONX_API_KEY = os.environ.get("WATSONX_API_KEY", "YOUR_IBM_CLOUD_API_KEY")
WATSONX_PROJECT_ID = os.environ.get("WATSONX_PROJECT_ID", "YOUR_WATSONX_PROJECT_ID")
WATSONX_URL = "https://us-south.ml.cloud.ibm.com"   # change region if needed
GRANITE_MODEL_ID = "ibm/granite-3-8b-instruct"       # mandatory IBM Granite model

print("Credentials configured (placeholders until filled in).")


## 2. Load the Knowledge Base

These are the same 4 documents uploaded to the agent's Knowledge (RAG) source in watsonx Orchestrate.

In [ ]:
KNOWLEDGE_FILES = [
    "01_Government_Schemes_India.txt",
    "02_Funding_Investor_Directory.txt",
    "03_Legal_Compliance_Checklist.txt",
    "04_BMC_Revenue_GTM_Templates.txt",
]

def load_knowledge_base(files):
    docs = {}
    for f in files:
        if os.path.exists(f):
            with open(f, "r", encoding="utf-8") as fh:
                docs[f] = fh.read()
        else:
            print(f"Warning: {f} not found in working directory.")
    return docs

knowledge_base = load_knowledge_base(KNOWLEDGE_FILES)
print(f"Loaded {len(knowledge_base)} knowledge documents:")
for name, text in knowledge_base.items():
    print(f"  - {name} ({len(text)} characters)")


## 3. Retrieval — Find Relevant Knowledge for the User's Idea

A lightweight keyword-overlap retriever splits each document into paragraphs and scores them against the user's startup idea, returning the top-matching chunks per document. This mirrors the retrieval step performed automatically by watsonx Orchestrate's built-in RAG pipeline.

In [ ]:
def chunk_document(text, min_len=40):
    """Split a document into paragraph-level chunks."""
    chunks = [c.strip() for c in text.split("\n\n") if len(c.strip()) > min_len]
    return chunks

def score_chunk(chunk, query_terms):
    chunk_lower = chunk.lower()
    return sum(1 for term in query_terms if term in chunk_lower)

def retrieve_relevant_context(user_idea, knowledge_base, top_k_per_doc=2):
    """Retrieve the most relevant chunks from each knowledge document for the user's idea."""
    query_terms = re.findall(r"[a-zA-Z]+", user_idea.lower())
    query_terms = [t for t in query_terms if len(t) > 3]  # drop small stopword-ish terms

    retrieved = []
    for doc_name, text in knowledge_base.items():
        chunks = chunk_document(text)
        scored = [(score_chunk(c, query_terms), c) for c in chunks]
        scored.sort(key=lambda x: x[0], reverse=True)
        top_chunks = [c for score, c in scored[:top_k_per_doc] if score > 0]
        if not top_chunks and chunks:
            top_chunks = chunks[:1]  # fallback: include at least one chunk for coverage
        for c in top_chunks:
            retrieved.append(f"[Source: {doc_name}]\n{c}")
    return "\n\n".join(retrieved)

# Example test
sample_idea = "An organic snack subscription box for working professionals in Pune"
context_preview = retrieve_relevant_context(sample_idea, knowledge_base)
print(context_preview[:1200], "...")


## 4. Behavior Instructions (System Prompt)

Identical to the Behavior instructions configured in the watsonx Orchestrate agent, enforcing the 9-section blueprint structure and grounding rules.

In [ ]:
SYSTEM_INSTRUCTIONS = """You are a Startup Blueprint Generator, an AI advisor that transforms a user's startup idea into a complete, structured business blueprint using retrieved knowledge on markets, funding, legal requirements, and government schemes.

When a user describes a startup idea, always respond with a structured blueprint containing these sections in order:
1. Idea Summary & Problem-Solution Fit
2. Business Model Canvas (all 9 blocks, tailored to the specific idea)
3. Market & Competitor Snapshot
4. Estimated Budget (6-month runway, broken into categories)
5. Revenue Model (recommend the most fitting model(s) with reasoning)
6. Go-to-Market Strategy (following the 5-step framework: target segment, positioning, channels, launch plan, early metrics)
7. Relevant Government Schemes & Incubators (cite specific schemes from your knowledge base)
8. Legal & Compliance Checklist (registration type recommendation, key steps)
9. Potential Investor Types & Funding Stage Guidance

Always ground funding, legal, and scheme information in the provided knowledge context rather than generating generic or fabricated details. If the user's idea is vague, ask 1-2 clarifying questions before generating the full blueprint. Keep tone professional, encouraging, and practical for first-time entrepreneurs.

GUIDELINES (guardrails):
- Do not fabricate specific scheme amounts, investor names, or legal requirements not present in the provided context.
- Keep responses structured with clear section headers; avoid unnecessary repetition.
- If asked something unrelated to startups/business planning, politely redirect to the blueprint purpose.
- Clarify that legal/financial guidance is general information, not certified advice.
"""
print("System instructions loaded.")


## 5. Generate the Blueprint using IBM Granite (watsonx.ai)

This calls the mandatory **IBM Granite** model hosted on **watsonx.ai** to generate the final grounded, structured business blueprint from the retrieved knowledge context.

In [ ]:
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

def get_granite_model():
    credentials = Credentials(
        url=WATSONX_URL,
        api_key=WATSONX_API_KEY,
    )
    model = ModelInference(
        model_id=GRANITE_MODEL_ID,
        credentials=credentials,
        project_id=WATSONX_PROJECT_ID,
        params={
            "decoding_method": "greedy",
            "max_new_tokens": 1500,
            "repetition_penalty": 1.05,
        },
    )
    return model

def generate_blueprint(user_idea, knowledge_base):
    context = retrieve_relevant_context(user_idea, knowledge_base)

    prompt = f"""{SYSTEM_INSTRUCTIONS}

RETRIEVED KNOWLEDGE CONTEXT:
{context}

USER'S STARTUP IDEA:
{user_idea}

Now generate the complete 9-section startup blueprint for this idea, grounded in the retrieved knowledge context above.
"""

    model = get_granite_model()
    response = model.generate_text(prompt=prompt)
    return response

# --- Run it ---
# user_idea = "An organic snack subscription box for working professionals in Pune"
# blueprint = generate_blueprint(user_idea, knowledge_base)
# print(blueprint)

print("Function ready. Fill in WATSONX_API_KEY and WATSONX_PROJECT_ID above, then call generate_blueprint(your_idea, knowledge_base).")


## 6. Example Run

Uncomment and run this cell once your credentials are filled in above.

In [ ]:
user_idea = "I have an idea for an app that connects college students with verified part-time tutors near their campus."

# blueprint = generate_blueprint(user_idea, knowledge_base)
# print(blueprint)

print("Ready to test — uncomment the two lines above once WATSONX_API_KEY / WATSONX_PROJECT_ID are set.")


## Notes

- This notebook implements the **same RAG + IBM Granite architecture** as the deployed watsonx Orchestrate agent (`Startup Blueprint Generator AI`), demonstrated here as standalone, inspectable Python code.
- The deployed agent itself runs live inside **IBM watsonx Orchestrate** (Agent Builder), where Knowledge, Behavior, Guidelines, and Toolset are configured through the no-code UI — this notebook exposes the equivalent logic in code form as required for submission.
- Retrieval here uses a simple keyword-overlap method for transparency; watsonx Orchestrate's built-in Knowledge (RAG) feature uses a more advanced embedding-based retrieval pipeline internally.
- Model: **IBM Granite (granite-3-8b-instruct)** via **watsonx.ai**, satisfying the mandatory technology requirement.
